In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('csv_checkpoints/training_log_ep380.csv')
df.info()

In [ ]:
# Success Rate per Scenario
fig, ax = plt.subplots(figsize=(12, 6))

scenario_stats = df.groupby('scenario').agg({
    'success': ['sum', 'count']
}).reset_index()
scenario_stats.columns = ['scenario', 'successes', 'total']
scenario_stats['success_rate'] = (scenario_stats['successes'] / scenario_stats['total']) * 100
scenario_stats = scenario_stats.sort_values('success_rate', ascending=False)

colors = ['green' if x >= 50 else 'orange' for x in scenario_stats['success_rate']]
bars = ax.barh(scenario_stats['scenario'], scenario_stats['success_rate'], color=colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Success Rate (%)')
ax.set_title('Success Rate per Vignette')
ax.set_xlim([0, 105])
ax.grid(True, alpha=0.3, axis='x')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, scenario_stats['success_rate'])):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2, f'{val:.1f}% ({int(scenario_stats.iloc[i]["successes"])}/{int(scenario_stats.iloc[i]["total"])})', 
            va='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Display as table
print("\n" + "=" * 80)
print("SUCCESS RATE BY VIGNETTE")
print("=" * 80)
print(scenario_stats.to_string(index=False))

In [ ]:
# Success Rate per Episode with 5% Tick Increments
plt.figure(figsize=(14, 4))
plt.plot(df['episode'], df['success_rate'] * 100, marker='o', linestyle='-', markersize=2, color='tab:blue')
for boundary in range(int((df['episode'].iloc[0] // 100 + 1) * 100), int(df['episode'].iloc[-1]) + 1, 100):
    plt.axvline(x=boundary, color='gray', linestyle='--', linewidth=1, alpha=0.5)
plt.xlabel('Episode')
plt.ylabel('Success Rate (%)')
plt.title('Success Rate per Episode')
plt.yticks(np.arange(0, 101, 5))
plt.ylim([-2, 102])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("OVERALL SUCCESS RATE ")
print("=" * 80)
overall_success_rate = (df['success'].sum() / df['success'].count()) * 100
print(f"Overall Success Rate: {overall_success_rate:.2f}% ({int(df['success'].sum())}/{int(df['success'].count())})")

In [ ]:
# Most Used Actions Analysis
print("=" * 80)
print("MOST USED ACTIONS ANALYSIS")
print("=" * 80)

# Most used action per episode (first 10 episodes)
print("\nMost Used Action per Episode (first 10 episodes):")
print("-" * 50)
for idx, row in df.head(10).iterrows():
    print(f"Episode {int(row['episode'])}: {row['most_used_action']}")

print(f"\n... (showing first 10 of {len(df)} episodes)")

# Overall most used actions across all episodes
print("\n" + "=" * 60)
print("OVERALL MOST USED ACTIONS ACROSS ALL EPISODES")
print("=" * 60)

action_columns = [f'action_{i}_count' for i in range(10)]
action_totals = df[action_columns].sum()

print("Total usage across all episodes:")
for i, total in enumerate(action_totals):
    percentage = (total / action_totals.sum()) * 100
    print(f"  action_{i}: {int(total)} times ({percentage:.1f}%)")

most_used_idx = action_totals.idxmax()
print(f"\nMost used action overall: {most_used_idx} ({int(action_totals.max())} times)")

# Visualize overall action usage
plt.figure(figsize=(12, 6))
bars = plt.bar(range(10), action_totals, color='lightcoral', edgecolor='darkred', linewidth=1.5)
plt.xlabel('Action Index')
plt.ylabel('Total Usage Count')
plt.title('Overall Action Usage Across All Episodes')
plt.xticks(range(10), [f'action_{i}' for i in range(10)], rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, action_totals)):
    plt.text(bar.get_x() + bar.get_width()/2, val + 50, f'{int(val)}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Average Time per Scenario
fig, ax = plt.subplots(figsize=(12, 6))

time_per_scenario = df.groupby('scenario')['duration_s'].agg(['max','mean', 'min', 'std', 'count']).reset_index()
time_per_scenario = time_per_scenario.sort_values('mean', ascending=False)

bars = ax.bar(range(len(time_per_scenario)), time_per_scenario['mean'], 
              color='mediumpurple', edgecolor='purple', linewidth=1.5,
              yerr=time_per_scenario['std'], capsize=5)
ax.set_ylabel('Average Duration (seconds)')
ax.set_xlabel('Vignette')
ax.set_title('Average Time per Vignette')
ax.set_xticks(range(len(time_per_scenario)))
ax.set_xticklabels(time_per_scenario['scenario'], rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, time_per_scenario['mean'])):
    ax.text(bar.get_x() + bar.get_width()/2, val + 2, f'{val:.1f}s', 
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Display as table
print("\n" + "=" * 80)
print("AVERAGE TIME BY VIGNETTE")
print("=" * 80)
print(time_per_scenario.to_string(index=False))

In [ ]:
import re

LOG_FILE = "training_20260527_09-04-28.txt"  

LOSS_PATTERNS = {
    "time_limit":        re.compile(r"\[TERMINAL\].*time limit expired", re.IGNORECASE),
    "detected_by_enemy": re.compile(r"\[TERMINAL\].*detected by enemy radar!", re.IGNORECASE),
}

counts     = {key: 0  for key in LOSS_PATTERNS}
loss_lines = {key: [] for key in LOSS_PATTERNS}

current_scenario = None

with open(LOG_FILE, encoding="utf-8", errors="replace") as f:
    for lineno, raw_line in enumerate(f, start=1):
        line = raw_line.strip()
        m = re.search(r"\[RESTART\] Scenario: (.+)", line)
        if m:
            current_scenario = m.group(1).strip()
        if current_scenario is None or "LandStrike" not in current_scenario:
            continue
        for key, pattern in LOSS_PATTERNS.items():
            if pattern.search(line):
                counts[key] += 1
                loss_lines[key].append((lineno, line))


print("=" * 60)
print("  LandStrike Loss Summary")
print("=" * 60)
print(f"  Time limit expired  : {counts['time_limit']:>4d} episode(s)")
print(f"  Detected by enemy   : {counts['detected_by_enemy']:>4d} episode(s)")
print("-" * 60)
print(f"  Total losses        : {sum(counts.values()):>4d} episode(s)")
print("=" * 60)


In [ ]:
import re

LOG_FILE = "training_20260527_09-04-28.txt"   # <-- update path if needed

LOSS_PATTERNS = {
    "Ally_hit":        re.compile(r"\[TERMINAL\].*Ally was hit", re.IGNORECASE),
    "Non_hostile": re.compile(r"\[TERMINAL\].*Non-hostile", re.IGNORECASE),
    "Merchant_reached_destination": re.compile(r"\[TERMINAL\].*Merchant reached destination", re.IGNORECASE),
}

counts     = {key: 0  for key in LOSS_PATTERNS}
loss_lines = {key: [] for key in LOSS_PATTERNS}

current_scenario = None

with open(LOG_FILE, encoding="utf-8", errors="replace") as f:
    for lineno, raw_line in enumerate(f, start=1):
        line = raw_line.strip()
        m = re.search(r"\[RESTART\] Scenario: (.+)", line)
        if m:
            current_scenario = m.group(1).strip()
        if current_scenario is None or "Escort" not in current_scenario:
            continue
        for key, pattern in LOSS_PATTERNS.items():
            if pattern.search(line):
                counts[key] += 1
                loss_lines[key].append((lineno, line))

# ── Results ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("  Escort Loss Summary")
print("=" * 60)
print(f"  Ally hit            : {counts['Ally_hit']:>4d} episode(s)")
print(f"  Non-hostile         : {counts['Non_hostile']:>4d} episode(s)")
print(f"  Merchant reached destination : {counts['Merchant_reached_destination']:>4d} episode(s)")
print("-" * 60)
print(f"  Total losses        : {sum(counts.values()):>4d} episode(s)")
print("=" * 60)